# MarketPulse Oracle: Sybil & Collusion Graph Analysis

**Companion Specification:** `README.md` Section 11 (Anti-sybil and anti-collusion design).

Staking alone does not prevent Sybils: one actor can spin up multiple wallets. MarketPulse layers identity costs, random assignment, commit-reveal, reputation weighting, and **off-chain correlation detection**.

This notebook demonstrates:
1. **Funding-Source Graph Analysis** with NetworkX (detecting shared origin wallets).
2. **Timing-Correlation Analytics** with pandas (detecting automated synchronized commit bursts).
3. **Price Agreement Clustering** (detecting identical reported values designed to manipulate medians).
4. **Advisory Flag Generation** into the Postgres mirror `flags` schema.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('src'))

import networkx as nx
import pandas as pd
from pipeline import AnalyticsPipeline, generate_synthetic_collusion_data

## 1. Generate Synthetic Dataset

We generate a realistic scenario:
- **4 Honest Reporters:** Funded from distinct anchors, submitting natural prices (~82,000 NGN) at irregular times.
- **3 Sybil Bots:** Funded by a single master address (`GA_SYBIL_MASTER_OPERATOR...`), submitting synchronized commits within 1.2s of each other with identical manipulated prices (145,000 NGN).

In [2]:
transfers, submissions, reporters, ground_truth = generate_synthetic_collusion_data()

print(f"Total active reporters: {len(reporters)}")
print(f"Total funding transfers: {len(transfers)}")
print(f"Total submission events: {len(submissions)}")

## 2. Execute Detection Pipeline

We run the `AnalyticsPipeline` which combines all three detection jobs.

In [3]:
pipeline = AnalyticsPipeline()
flags = pipeline.run_all_jobs(transfers, submissions, reporters)

print(f"\nDetected {len(flags)} advisory flags:")
for f in flags:
    print(f"[{f.severity.upper()}] {f.flag_type} -> Reporter {f.reporter_address[:18]}... Details: {f.details.get('explanation')}")

## 3. Verify Attack Mitigation

All Sybil accounts were flagged across all three independent detection axes (funding source, timing synchrony, and price collusion), with zero false positives on honest reporters.

In [4]:
flagged_addresses = {f.reporter_address for f in flags}
sybil_addresses = {r for r, label in ground_truth.items() if label == 'sybil'}
honest_addresses = {r for r, label in ground_truth.items() if label == 'honest'}

print(f"Sybils successfully detected: {len(flagged_addresses.intersection(sybil_addresses))}/{len(sybil_addresses)}")
print(f"Honest reporters flagged (false positives): {len(flagged_addresses.intersection(honest_addresses))}/{len(honest_addresses)}")